In [1]:
pip install flask

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install flask pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from flask import Flask, request, jsonify
import pymysql

app = Flask(__name__)

# koneksi manual
def get_db_connection():
    return pymysql.connect(
        host="localhost",
        user="root",
        password="",
        database="wp1",
        cursorclass=pymysql.cursors.DictCursor
    )


@app.route("/categories", methods=["GET"])
def get_all_categories():
    conn = get_db_connection()
    cur = conn.cursor()

    cur.execute("SELECT id, name FROM categories ORDER BY id ASC")
    rows = cur.fetchall()

    cur.close()
    conn.close()

    return jsonify({
        "success": True,
        "data": rows
    })


@app.route("/categories/<int:id>", methods=["GET"])
def get_by_id(id):
    conn = get_db_connection()
    cur = conn.cursor()

    cur.execute("SELECT id, name FROM categories WHERE id=%s", (id,))
    row = cur.fetchone()

    cur.close()
    conn.close()

    if row:
        return jsonify({"success": True, "data": row})

    return jsonify({"success": False, "message": "Not found"}), 404


@app.route("/categories", methods=["POST"])
def insert():
    data = request.get_json()
    name = data.get("name")

    conn = get_db_connection()
    cur = conn.cursor()

    cur.execute("INSERT INTO categories (name) VALUES (%s)", (name,))
    conn.commit()

    new_id = cur.lastrowid

    cur.close()
    conn.close()

    return jsonify({
        "success": True,
        "data": {
            "id": new_id,
            "name": name
        }
    }), 201


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=8000, debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8000
Press CTRL+C to quit
127.0.0.1 - - [15/Apr/2026 09:58:08] "GET /categories HTTP/1.1" 500 -
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/pymysql/connections.py", line 661, in connect
    sock = socket.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/socket.py", line 851, in create_connection
    raise exceptions[0]
  File "/opt/anaconda3/lib/python3.11/socket.py", line 836, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 61] Connection refused

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/flask/app.py", line 2552, in __call__
    return self.wsgi_app(environ, start_response)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/site-packages/flask/app.py", line 2532, in wsgi_app
    response = self.